# Operational Solar Dataset Exploration

This notebook explores version 1 of the [Solar Power Dataset](https://www.kaggle.com/datasets/s1nister/solar-power-generation-dataset), licensed CC0 / Public Domain. It performs descriptive analysis only—no anomaly detection or anomaly removal.

**Timezone:** Not specified by the dataset source. Parsed timestamps remain timezone-naive.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / 'src').is_dir():
    if (PROJECT_ROOT.parent / 'src').is_dir():
        PROJECT_ROOT = PROJECT_ROOT.parent
    else:
        raise RuntimeError('Run this notebook from the project root or notebooks directory.')

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.data_processing.analyze_operational_data import analyze_csv
from src.data_processing.load_operational_data import (
    DEFAULT_TIMESTAMP_FORMAT,
    build_timestamp_profile,
    discover_csv_files,
    load_operational_csv,
)

DATASET_DIR = PROJECT_ROOT / 'data' / 'operational' / 'solar-power-dataset'
csv_files = discover_csv_files(DATASET_DIR)
csv_files

## Load the source CSV

Load the raw file separately for faithful inspection. The reusable loader is applied afterward and does not modify the source file.

In [ ]:
if len(csv_files) != 1:
    raise RuntimeError(f'Expected one downloaded CSV, found {len(csv_files)}: {csv_files}')

csv_path = csv_files[0]
raw = pd.read_csv(csv_path)
print(f'File: {csv_path.name}')
print(f'Shape: {raw.shape[0]} rows x {raw.shape[1]} columns')
display(raw.head())

## Schema, missing values, and duplicates

In [ ]:
schema = pd.DataFrame({
    'dtype': raw.dtypes.astype(str),
    'missing_count': raw.isna().sum(),
    'unique_count': raw.nunique(dropna=False),
})
display(schema)
print(f'Exact duplicate rows: {int(raw.duplicated().sum())}')

## Conservative loading and timestamp checks

The observed source format is `%d-%m-%Y %H:%M`. Parsing does not assign a timezone.

In [ ]:
data = load_operational_csv(csv_path, drop_duplicate_rows=False)
timestamp_profile = build_timestamp_profile(
    raw['Timestamp'], timestamp_format=DEFAULT_TIMESTAMP_FORMAT
)
display(pd.Series(timestamp_profile, name='value').to_frame())
print(f"Parsed dtype: {data['Timestamp'].dtype}")
print(f"Timezone: {data['Timestamp'].dt.tz}")

## Descriptive statistics

In [ ]:
display(data.select_dtypes(include='number').describe().T)

## Data-quality flags

These are review flags only. The source rows remain unchanged and no flagged value is automatically classified as an anomaly.

In [ ]:
quality = analyze_csv(csv_path)
flags = pd.DataFrame(quality['physical_range_flags']).T
display(flags[['rule', 'count', 'minimum_flagged_value', 'maximum_flagged_value']])
print(f"Invalid non-numeric values: {quality['total_invalid_non_numeric_values']}")

## Measurements over time

In [ ]:
time_plots = [
    ('Power_Generated', 'Generated Power over Time', 'Power generated (W)'),
    ('Solar_Radiation', 'Solar Radiation over Time', 'Solar radiation (W/m²)'),
    ('Air_Temp', 'Air Temperature over Time', 'Air temperature (°C)'),
]
fig, axes = plt.subplots(len(time_plots), 1, figsize=(14, 10), sharex=True)
for axis, (column, title, ylabel) in zip(axes, time_plots):
    axis.plot(data['Timestamp'], data[column], linewidth=1)
    axis.set(title=title, ylabel=ylabel)
    axis.grid(alpha=0.25)
axes[-1].set_xlabel('Timestamp (timezone not specified)')
plt.tight_layout()
plt.show()

## Power/radiation relationship and correlations

In [ ]:
numeric = data.select_dtypes(include='number')
correlations = numeric.corr()

fig, axes = plt.subplots(1, 2, figsize=(17, 6))
axes[0].scatter(data['Solar_Radiation'], data['Power_Generated'], s=14, alpha=0.55)
axes[0].set(
    title='Generated Power versus Solar Radiation',
    xlabel='Solar radiation (W/m²)',
    ylabel='Power generated (W)',
)
axes[0].grid(alpha=0.25)

image = axes[1].imshow(correlations, cmap='coolwarm', vmin=-1, vmax=1)
axes[1].set_xticks(np.arange(len(correlations.columns)))
axes[1].set_yticks(np.arange(len(correlations.columns)))
axes[1].set_xticklabels(correlations.columns, rotation=60, ha='right')
axes[1].set_yticklabels(correlations.columns)
axes[1].set_title('Pearson Correlation Matrix')
fig.colorbar(image, ax=axes[1], label='Pearson correlation')
plt.tight_layout()
plt.show()
display(correlations['Power_Generated'].sort_values(ascending=False).to_frame())

## Daily power-production pattern

The file contains only one complete calendar day plus two partial days, so this view is descriptive rather than a long-term daily profile.

In [ ]:
daily = data[['Timestamp', 'Power_Generated']].dropna().copy()
daily['minute_of_day'] = daily['Timestamp'].dt.hour * 60 + daily['Timestamp'].dt.minute
daily_profile = daily.groupby('minute_of_day')['Power_Generated'].mean()

fig, axis = plt.subplots(figsize=(11, 4.5))
axis.plot(daily_profile.index / 60, daily_profile.values, linewidth=1.5)
axis.set(
    title='Mean Generated Power by Time of Day',
    xlabel='Time of day (hours; timezone not specified)',
    ylabel='Power generated (W)',
    xlim=(0, 24),
)
axis.grid(alpha=0.25)
plt.show()

## Observations from downloaded version 1

- `solar_data.csv` contains 1,009 rows and 14 columns from 2022-04-27 15:32 through 2022-04-29 01:08.
- All 1,008 consecutive timestamp differences are exactly 120 seconds. There are no timestamp parse failures, ordering reversals, duplicate timestamps, or gaps.
- No missing values, exact duplicate rows, or invalid numeric strings were found.
- 314 solar-radiation readings are below 0 W/m² (minimum -0.838326), and 10 wind-direction readings exceed 360 degrees (maximum 385.630880). They are retained as quality flags, not labeled anomalies.
- Generated power ranges from 8.485232 to 438.556840 W and has Pearson correlation about 0.730 with solar radiation in this short sample.
- Generated power is almost perfectly correlated with array voltage (about 0.99999), and voltage multiplied by current reproduces power to rounding precision. This deterministic relationship will require care to avoid target leakage later.
- The time span is too short for seasonal or long-term conclusions.

These observations apply only to the downloaded file identified in `docs/operational_dataset.md`. Rerun the notebook and quality script if the source version changes.